<a href="https://colab.research.google.com/github/rotoncsedu/DL/blob/main/MNIST_by_hand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import tensorflow as tf
class NaiveDense:
  def __init__(self, input_size, output_size, activation):
    self.activation = activation
## Create matrix W and initialize with random values
    w_shape = (input_size, output_size)
    w_initial_value = tf.random.uniform(w_shape,minval = 0, maxval = 1e-1)
    self.W = tf.Variable(w_initial_value)
## Create vector b and intialize with 0
    b_shape = (output_size,)
    b_initial_value = tf.zeros(b_shape)
    self.b = tf.Variable(b_initial_value)

  def __call__(self, inputs):
    return self.activation(tf.matmul(inputs, self.W) + self.b)

  def weights(self):
    return [self.W, self.b]



In [25]:
class NaiveSequential:
  def __init__(self,layers):
    self.layers = layers

  def __call__(self,inputs):
    x = inputs
    for layer in self.layers:
      x = layer(x)
    return x

  def weights(self):
    weights = []
    for layer in self.layers:
      weights += layer.weights()
    return weights

In [26]:
model  = NaiveSequential([
    NaiveDense(input_size= 28*28, output_size= 512, activation= tf.nn.relu),
    NaiveDense(input_size= 512, output_size= 10, activation= tf.nn.softmax)
])

In [27]:
import math
class BatchGenerator:
  def __init__(self,images, labels, batch_size = 128):
    assert len(images) == len(labels)
    self.index = 0
    self.images = images
    self.labels = labels
    self.batch_size = batch_size
    self.num_batches = math.ceil(len(images) / batch_size)

  def next(self):
    images = self.images[self.index : self.index + self.batch_size]
    labels = self.labels[self.index : self.index + self.batch_size]
    self.index += self.batch_size
    return images, labels

In [28]:
def one_training_step(model, images_batch, labels_batch):
  with tf.GradientTape() as tape:
    predictions = model(images_batch)
    per_sample_losses = tf.keras.losses.sparse_categorical_crossentropy(labels_batch, predictions)
    average_loss = tf.reduce_mean(per_sample_losses)

  gradients = tape.gradient(average_loss, model.weights())
  update_weights(gradients, model.weights())
  return average_loss

learning_rate = 1e-3
def update_weights(gradients, weights):
  for g,w in zip(gradients, weights):
    w.assign_sub(g * learning_rate)

In [29]:
def fit(model, images, labels, epochs, batch_size = 128):
  for epoch_counter in range(epochs):
    print(f"Epoch {epoch_counter}")
    batch_generator = BatchGenerator(images, labels)
    for batch_counter in range(batch_generator.num_batches):
      images_batch, labels_batch = batch_generator.next()
      loss = one_training_step(model, images_batch, labels_batch)
      if batch_counter % 100 == 0:
        print(f"loss at batch {batch_counter}: {loss:.2f}")

In [30]:
from tensorflow.keras.datasets import mnist
(train_img, train_labels), (test_img, test_labels) = mnist.load_data()

train_img = train_img.reshape((60000, 28*28))
train_img = train_img.astype("float32") / 255
test_img = test_img.reshape((10000, 28*28))
test_iimg = test_img.astype("float32") / 255

fit(model, train_img, train_labels, epochs= 10, batch_size= 128)

Epoch 0
loss at batch 0: 4.80
loss at batch 100: 2.26
loss at batch 200: 2.23
loss at batch 300: 2.12
loss at batch 400: 2.25
Epoch 1
loss at batch 0: 1.93
loss at batch 100: 1.90
loss at batch 200: 1.86
loss at batch 300: 1.75
loss at batch 400: 1.86
Epoch 2
loss at batch 0: 1.61
loss at batch 100: 1.60
loss at batch 200: 1.54
loss at batch 300: 1.45
loss at batch 400: 1.53
Epoch 3
loss at batch 0: 1.35
loss at batch 100: 1.36
loss at batch 200: 1.26
loss at batch 300: 1.23
loss at batch 400: 1.29
Epoch 4
loss at batch 0: 1.14
loss at batch 100: 1.17
loss at batch 200: 1.05
loss at batch 300: 1.06
loss at batch 400: 1.12
Epoch 5
loss at batch 0: 0.99
loss at batch 100: 1.03
loss at batch 200: 0.91
loss at batch 300: 0.94
loss at batch 400: 0.99
Epoch 6
loss at batch 0: 0.88
loss at batch 100: 0.92
loss at batch 200: 0.80
loss at batch 300: 0.84
loss at batch 400: 0.90
Epoch 7
loss at batch 0: 0.80
loss at batch 100: 0.83
loss at batch 200: 0.72
loss at batch 300: 0.77
loss at batch 40

In [31]:
# Evaluate the model on the test data
def evaluate(model, images, labels, batch_size=128):
  batch_generator = BatchGenerator(images, labels, batch_size)
  correct_predictions = 0
  total_samples = 0

  for batch_counter in range(batch_generator.num_batches):
    images_batch, labels_batch = batch_generator.next()
    predictions = model(images_batch)
    # Get the predicted class (index with highest probability)
    predicted_classes = tf.argmax(predictions, axis=1)
    # Compare with true labels
    correct_predictions += tf.reduce_sum(tf.cast(tf.equal(predicted_classes, labels_batch), tf.float32))
    total_samples += len(labels_batch)

  return correct_predictions / total_samples

test_accuracy = evaluate(model, test_iimg, test_labels)
print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.8157
